Notebook 05 — Chatbot IA d'exploration des résultats======================================================Projet : Détection de biais dans les données de santé au travailAuteure : Laure Bonnefond — IPRP / IST — Master IA Data IPSSI BordeauxCe notebook met en place un chatbot conversationnel qui permet d'explorerles résultats du projet en langage naturel. L'assistant utilise l'APIAnthropic Claude avec un prompt système enrichi des résultats des 4notebooks précédents.Fonctionnalités :  - Prompt système enrichi du contexte projet  - Historique de conversation maintenu  - Avatar professionnel intégré  - Liens vers les sources fiables (INRS, DARES, Ameli)  - Interface Jupyter avec ipywidgets⚠️ Sécurité : la clé API doit être stockée dans une variable d'environnement,JAMAIS dans le code source ni dans le dépôt Git.

# 1. Configuration


In [ ]:
import os
import json
import pandas as pd
import numpy as np
from datetime import datetime
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Chargement du résumé du modèle ML (Notebook 04)
with open('../data/processed/04_resume_modele.json', 'r') as f:
    resume_modele = json.load(f)

# Chargement des résultats des notebooks précédents
df_biais = pd.read_csv('../data/processed/02_biais_critiques.csv')
df_correlations = pd.read_csv('../data/processed/03_top_correlations.csv')
df_features = pd.read_csv('../data/processed/04_feature_importance.csv')

print("Contexte projet chargé.")
print(f"  Biais critiques détectés     : {len(df_biais)}")
print(f"  Corrélations significatives  : {len(df_correlations)}")
print(f"  Accuracy modèle ML           : {resume_modele['accuracy_test']:.1%}")
print(f"  Écart fairness (équité)      : {resume_modele['ecart_fairness']:.1%}")


# 2. Construction du prompt système enrichi


In [ ]:
def construire_prompt_systeme():
    """Construit un prompt système enrichi des résultats du projet."""
    
    top_biais = df_biais.head(5).to_dict('records')
    top_correlations = df_correlations.head(5).to_dict('records')
    
    prompt = f"""Tu es l'assistant IA du projet "Détection de biais dans les données de santé au travail" développé par Laure Bonnefond, infirmière en santé au travail et IPRP, dans le cadre de son Master IA, Big Data & Développement à l'IPSSI Bordeaux.

# Ton rôle
Tu aides les visiteurs du portfolio (recruteurs, enseignants, collègues IPRP) à explorer les résultats du projet en langage naturel. Tu réponds de façon concise, factuelle et pédagogique. Tu n'inventes JAMAIS de chiffres : tu utilises uniquement les données ci-dessous.

# Contexte du projet
Le projet analyse la représentativité des données ouvertes de santé au travail (AT/MP de l'Assurance Maladie + expositions SUMER de la DARES) et identifie les biais qui pourraient fausser un modèle d'IA prédictive. Il s'appuie sur l'expertise terrain de 20 ans de Laure en santé au travail.

# Sources de données
- AT/MP : Assurance Maladie — open data Risques Professionnels
- SUMER : DARES — Enquête Surveillance médicale des expositions 2017
- INSEE : Population active de référence

# Résultats clés

## Notebook 02 — Détection des biais
- 9 secteurs sur 9 présentent un biais significatif au test du Chi² (p < 0.001)
- {len(df_biais)} biais critiques détectés (écart > 10%)
- Biais les plus marqués :
"""
    for b in top_biais:
        prompt += f"  • {b['secteur']} — {b['tranche_age']} : ratio {b['ratio']} ({b['type']}, écart {b['ecart_pct']:+.0f}%)\n"
    
    prompt += f"""
- Les 60+ ans sont systématiquement sous-représentés dans les secteurs industriels (effet "travailleur sain", reclassement, retraite anticipée)
- Les jeunes (18-29) sont sur-représentés dans les AT (inexpérience, prise de risque)
- Les femmes sont sous-représentées dans les AT/MP des secteurs industriels (postes administratifs moins exposés mais aussi sous-déclaration)

## Notebook 03 — Corrélations AT × DARES
- {len(df_correlations)} corrélations significatives (p < 0.05) sur 40 testées
- Top corrélations :
"""
    for c in top_correlations:
        prompt += f"  • {c['exposition']} → {c['outcome']} : r = {c['r']:+.2f} (p = {c['p_value']:.4f})\n"
    
    prompt += f"""
- Les contraintes posturales prédisent 82% de la variance de l'indice de fréquence AT
- Le travail sur écran > 6h est corrélé négativement avec les AT (effet protecteur des bureaux)
- Le secteur Transports/Eau/Gaz/Électricité est en sur-sinistralité (+14 points vs tendance)

## Notebook 04 — Modèle ML
- Algorithme : Random Forest Classifier (200 arbres, profondeur 8)
- Dataset : 151 codes NAF
- Accuracy test : {resume_modele['accuracy_test']:.1%}
- F1-macro : {resume_modele['f1_macro_test']:.3f}
- CV F1 (5 folds) : {resume_modele['cv_f1_mean']:.3f} ± {resume_modele['cv_f1_std']:.3f}
- Écart fairness (équité H vs Mixte) : {resume_modele['ecart_fairness']:.1%} — équité satisfaisante (seuil 5%)

Top features (SHAP) :
"""
    for _, f in df_features.head(5).iterrows():
        prompt += f"  • {f['feature']} : importance permutation {f['importance_perm']:+.3f}\n"
    
    prompt += """
# Style de réponse
- Concis (max 4-5 phrases sauf demande explicite)
- Cite les chiffres précis quand pertinent
- Reconnais les limites du projet (dataset agrégé, 9 CTN d'origine, biais déclaratif non corrigé)
- Renvoie vers les notebooks (01 à 05) ou les sources fiables (INRS, DARES, Ameli) quand utile
- Ne fais pas de diagnostic médical, ne donne pas de conseil juridique

# Sources fiables à citer si pertinent
- INRS (Institut National de Recherche et de Sécurité) : inrs.fr
- DARES (Direction Animation Recherche Études Statistiques) : dares.travail-emploi.gouv.fr  
- Ameli — Risques Professionnels : assurance-maladie.ameli.fr
- HAS (Haute Autorité de Santé) : has-sante.fr
- ANSES : anses.fr
"""
    return prompt

prompt_systeme = construire_prompt_systeme()
print(f"Prompt système construit : {len(prompt_systeme)} caractères")
print("\n--- EXTRAIT ---")
print(prompt_systeme[:800] + "\n...")


# 3. Classe Chatbot


In [ ]:
class ChatbotSanteTravail:
    """Chatbot d'exploration des résultats du projet."""
    
    def __init__(self, prompt_systeme, model="claude-sonnet-4-5"):
        self.prompt_systeme = prompt_systeme
        self.model = model
        self.historique = []
        self.api_key = os.environ.get('ANTHROPIC_API_KEY')
        
        if not self.api_key:
            print("⚠️  ATTENTION : variable d'environnement ANTHROPIC_API_KEY non définie")
            print("   En mode démo, les réponses seront simulées.")
            print("   Pour activer le chatbot : export ANTHROPIC_API_KEY='sk-ant-...'")
            self.mode_demo = True
        else:
            self.mode_demo = False
            try:
                import anthropic
                self.client = anthropic.Anthropic(api_key=self.api_key)
                print("✅ Client Anthropic initialisé")
            except ImportError:
                print("⚠️  Package anthropic non installé : pip install anthropic")
                self.mode_demo = True
    
    def repondre(self, question):
        """Envoie une question et retourne la réponse."""
        self.historique.append({"role": "user", "content": question})
        
        if self.mode_demo:
            reponse = self._reponse_demo(question)
        else:
            reponse = self._appel_api(question)
        
        self.historique.append({"role": "assistant", "content": reponse})
        return reponse
    
    def _appel_api(self, question):
        """Appel réel à l'API Anthropic."""
        try:
            response = self.client.messages.create(
                model=self.model,
                max_tokens=1024,
                system=self.prompt_systeme,
                messages=self.historique,
            )
            return response.content[0].text
        except Exception as e:
            return f"⚠️ Erreur API : {str(e)[:200]}"
    
    def _reponse_demo(self, question):
        """Réponses simulées pour la démo (sans API key)."""
        q = question.lower()
        
        if 'biais' in q and ('âge' in q or 'age' in q or 'senior' in q):
            return ("Le projet a détecté un biais d'âge systématique : les 60+ ans "
                    "sont sous-représentés dans les AT/MP de tous les secteurs "
                    "industriels (ratio 0.50 à 0.71). Métallurgie -50%, BTP -40%. "
                    "Causes probables : effet 'travailleur sain', reclassement, "
                    "retraite anticipée. Voir Notebook 02 pour le détail.")
        elif 'femme' in q or 'genre' in q or 'sexe' in q:
            return ("Les femmes sont sous-représentées dans les AT/MP des secteurs "
                    "industriels : Métallurgie (ratio 0.67), BTP (0.75), Transports "
                    "(0.79). Le test du Chi² confirme un biais significatif dans "
                    "9/9 secteurs. Causes : postes administratifs moins exposés, "
                    "mais aussi possible sous-déclaration.")
        elif 'corrélation' in q or 'predict' in q or 'risque' in q:
            return ("Les corrélations les plus fortes : contraintes posturales → "
                    "indice de fréquence AT (r=0.91, p<0.001), agents biologiques "
                    "→ MP (r=0.89), travail sur écran → AT (r=-0.86, effet "
                    "protecteur). Voir Notebook 03 pour la heatmap complète.")
        elif 'modèle' in q or 'ml' in q or 'machine learning' in q:
            return (f"Le modèle Random Forest atteint {resume_modele['accuracy_test']:.0%} "
                    f"d'accuracy et un F1-macro de {resume_modele['f1_macro_test']:.2f} "
                    "sur 151 codes NAF. L'audit fairness montre un écart de seulement "
                    f"{resume_modele['ecart_fairness']:.1%} entre secteurs masculins "
                    "et mixtes — le modèle est équitable. Voir Notebook 04.")
        elif 'source' in q or 'données' in q:
            return ("Le projet utilise 3 sources ouvertes : Ameli (open data AT/MP "
                    "par CTN), DARES (enquête SUMER 2017 sur les expositions), et "
                    "INSEE (population active de référence). Liens dans le README.")
        elif 'limite' in q:
            return ("Limites principales : (1) dataset agrégé au niveau secteur — "
                    "pas d'analyse par entreprise; (2) le biais déclaratif n'est pas "
                    "corrigé — un secteur sous-déclarant apparaît à tort 'à faible "
                    "risque'; (3) 9 CTN d'origine — généralisation limitée à MSA/DOM.")
        else:
            return ("Je peux te renseigner sur : les biais détectés (âge, genre), "
                    "les corrélations AT × expositions, le modèle ML et son audit "
                    "fairness, les sources de données, ou les limites du projet. "
                    "Quelle dimension t'intéresse ?")
    
    def reset(self):
        """Efface l'historique de conversation."""
        self.historique = []
        print("Historique effacé.")


# 4. Test du chatbot


In [ ]:
chatbot = ChatbotSanteTravail(prompt_systeme)

# Questions de test (réponses simulées en mode démo)
questions_test = [
    "Quels sont les principaux biais détectés ?",
    "Le modèle ML est-il équitable selon le sexe ?",
    "Quelles sont les corrélations les plus fortes ?",
    "Quelles sont les limites du projet ?",
]

print("=" * 70)
print("DÉMONSTRATION DU CHATBOT")
print("=" * 70)

for q in questions_test:
    print(f"\n👤 Question : {q}")
    reponse = chatbot.repondre(q)
    print(f"🤖 Assistant : {reponse}\n")
    print("-" * 70)


# 5. Interface interactive (Jupyter widget)

Le code ci-dessous active une interface interactive si vous êtes dans
Jupyter Notebook ou JupyterLab. Sinon, utilisez la fonction `chatbot.repondre()`.


In [ ]:
def lancer_interface_interactive():
    """Lance une interface interactive avec ipywidgets (Jupyter uniquement)."""
    try:
        import ipywidgets as widgets
        from IPython.display import display, HTML, clear_output
    except ImportError:
        print("⚠️  ipywidgets non installé. Installez avec : pip install ipywidgets")
        return None
    
    # Avatar (SVG inline pour le portfolio)
    avatar_svg = '''
    <svg width="60" height="60" viewBox="0 0 60 60" xmlns="http://www.w3.org/2000/svg">
        <circle cx="30" cy="30" r="28" fill="#0F6E56" stroke="#0a4a3a" stroke-width="2"/>
        <text x="30" y="38" text-anchor="middle" fill="white" 
              font-size="24" font-weight="bold" font-family="Arial">LB</text>
    </svg>
    '''
    
    output = widgets.Output()
    input_text = widgets.Text(
        placeholder='Posez votre question sur le projet...',
        layout=widgets.Layout(width='80%')
    )
    bouton_envoyer = widgets.Button(description='Envoyer', button_style='success')
    bouton_reset = widgets.Button(description='Réinitialiser', button_style='warning')
    
    def on_envoyer(b):
        with output:
            question = input_text.value
            if not question.strip():
                return
            print(f"\n👤 {question}\n")
            reponse = chatbot.repondre(question)
            display(HTML(f'<div style="display:flex;align-items:flex-start;'
                          f'margin:10px 0;padding:10px;background:#f0f7f4;'
                          f'border-radius:8px;">'
                          f'<div style="margin-right:10px;">{avatar_svg}</div>'
                          f'<div><b>Laure (assistant IA)</b><br>{reponse}</div>'
                          f'</div>'))
            input_text.value = ''
    
    def on_reset(b):
        with output:
            clear_output()
            chatbot.reset()
            print("✓ Conversation réinitialisée")
    
    bouton_envoyer.on_click(on_envoyer)
    bouton_reset.on_click(on_reset)
    input_text.on_submit(on_envoyer)
    
    # En-tête avec avatar
    header = widgets.HTML(
        f'<div style="display:flex;align-items:center;padding:15px;'
        f'background:#0F6E56;color:white;border-radius:8px 8px 0 0;">'
        f'<div style="margin-right:15px;">{avatar_svg}</div>'
        f'<div><h3 style="margin:0;color:white;">Chatbot Santé au Travail</h3>'
        f'<small>Projet portfolio — Laure Bonnefond, IPRP & Master IA IPSSI</small>'
        f'</div></div>'
    )
    
    interface = widgets.VBox([
        header,
        widgets.HBox([input_text, bouton_envoyer]),
        bouton_reset,
        output,
    ])
    display(interface)
    return interface

# Pour lancer l'interface dans Jupyter :
# lancer_interface_interactive()


# 6. Génération d'un widget HTML autonome

On crée une version HTML/JS du chatbot que tu peux héberger sur GitHub
Pages — c'est l'élément que les recruteurs verront en visitant ton portfolio.


In [ ]:
def generer_widget_html():
    """Génère un widget HTML autonome avec interface chatbot."""
    
    # Préparer les données du contexte projet en JSON
    contexte = {
        'projet': 'Détection de biais dans les données de santé au travail',
        'auteur': 'Laure Bonnefond — IPRP & Master IA IPSSI',
        'biais_critiques': len(df_biais),
        'correlations_significatives': len(df_correlations),
        'accuracy_modele': f"{resume_modele['accuracy_test']:.1%}",
        'fairness': f"{resume_modele['ecart_fairness']:.1%}",
        'top_biais': df_biais.head(3).to_dict('records'),
        'top_correlations': df_correlations.head(3).to_dict('records'),
    }
    
    contexte_json = json.dumps(contexte, ensure_ascii=False, indent=2)
    
    html = '''<!DOCTYPE html>
<html lang="fr">
<head>
<meta charset="UTF-8">
<title>Chatbot — Biais Santé au Travail</title>
<style>
  * { box-sizing: border-box; }
  body {
    font-family: 'Inter', -apple-system, BlinkMacSystemFont, sans-serif;
    margin: 0;
    background: linear-gradient(135deg, #f0f7f4 0%, #e3f0eb 100%);
    min-height: 100vh;
    padding: 20px;
  }
  .chatbot-container {
    max-width: 720px;
    margin: 0 auto;
    background: white;
    border-radius: 16px;
    box-shadow: 0 8px 32px rgba(15, 110, 86, 0.15);
    overflow: hidden;
  }
  .header {
    background: linear-gradient(135deg, #0F6E56 0%, #185FA5 100%);
    color: white;
    padding: 24px;
    display: flex;
    align-items: center;
    gap: 16px;
  }
  .avatar {
    width: 56px; height: 56px;
    border-radius: 50%;
    background: white;
    color: #0F6E56;
    display: flex;
    align-items: center;
    justify-content: center;
    font-weight: bold;
    font-size: 20px;
    border: 3px solid rgba(255,255,255,0.3);
  }
  .header h1 { margin: 0; font-size: 18px; }
  .header p { margin: 4px 0 0; opacity: 0.9; font-size: 13px; }
  .chat-area {
    height: 420px;
    overflow-y: auto;
    padding: 20px;
    background: #fafbfc;
  }
  .message {
    margin: 12px 0;
    display: flex;
    gap: 10px;
    animation: fadeIn 0.3s;
  }
  @keyframes fadeIn {
    from { opacity: 0; transform: translateY(8px); }
    to { opacity: 1; transform: translateY(0); }
  }
  .message-bot { justify-content: flex-start; }
  .message-user { justify-content: flex-end; }
  .bubble {
    max-width: 75%;
    padding: 12px 16px;
    border-radius: 16px;
    font-size: 14px;
    line-height: 1.5;
  }
  .bubble-bot {
    background: white;
    border: 1px solid #e3f0eb;
    color: #1a2e26;
    border-radius: 16px 16px 16px 4px;
  }
  .bubble-user {
    background: linear-gradient(135deg, #0F6E56 0%, #185FA5 100%);
    color: white;
    border-radius: 16px 16px 4px 16px;
  }
  .mini-avatar {
    width: 32px; height: 32px;
    border-radius: 50%;
    background: #0F6E56;
    color: white;
    display: flex;
    align-items: center;
    justify-content: center;
    font-weight: bold;
    font-size: 12px;
    flex-shrink: 0;
  }
  .input-area {
    padding: 16px;
    background: white;
    border-top: 1px solid #e3f0eb;
    display: flex;
    gap: 8px;
  }
  .input-area input {
    flex: 1;
    padding: 12px 16px;
    border: 1px solid #d4e8e0;
    border-radius: 12px;
    font-size: 14px;
    outline: none;
  }
  .input-area input:focus { border-color: #0F6E56; }
  .input-area button {
    padding: 12px 24px;
    background: linear-gradient(135deg, #0F6E56 0%, #185FA5 100%);
    color: white;
    border: none;
    border-radius: 12px;
    cursor: pointer;
    font-weight: 600;
    font-size: 14px;
  }
  .suggestions {
    padding: 12px 20px;
    background: #fafbfc;
    border-top: 1px solid #e3f0eb;
    display: flex;
    flex-wrap: wrap;
    gap: 8px;
  }
  .suggestion-chip {
    padding: 6px 12px;
    background: white;
    border: 1px solid #d4e8e0;
    border-radius: 16px;
    font-size: 12px;
    cursor: pointer;
    color: #0F6E56;
  }
  .suggestion-chip:hover { background: #e3f0eb; }
  .sources {
    padding: 16px 20px;
    background: #f0f7f4;
    font-size: 12px;
    color: #5a6f68;
    border-top: 1px solid #e3f0eb;
  }
  .sources a {
    color: #0F6E56;
    text-decoration: none;
    margin-right: 12px;
  }
  .sources a:hover { text-decoration: underline; }
</style>
</head>
<body>

<div class="chatbot-container">
  <div class="header">
    <div class="avatar">LB</div>
    <div>
      <h1>Chatbot — Projet Biais Santé au Travail</h1>
      <p>Laure Bonnefond, IPRP &amp; Master IA IPSSI Bordeaux</p>
    </div>
  </div>
  
  <div class="chat-area" id="chat">
    <div class="message message-bot">
      <div class="mini-avatar">LB</div>
      <div class="bubble bubble-bot">
        Bonjour ! Je suis l'assistant IA du projet de détection de biais dans
        les données de santé au travail. Posez-moi des questions sur les biais
        détectés, les corrélations AT × DARES, le modèle ML, ou les sources
        de données.
      </div>
    </div>
  </div>
  
  <div class="suggestions">
    <div class="suggestion-chip" onclick="poserQuestion('Quels biais avez-vous détectés ?')">Quels biais ?</div>
    <div class="suggestion-chip" onclick="poserQuestion('Comment fonctionne le modèle ML ?')">Le modèle ML</div>
    <div class="suggestion-chip" onclick="poserQuestion('Quelles corrélations significatives ?')">Corrélations</div>
    <div class="suggestion-chip" onclick="poserQuestion('Quelles sont les limites du projet ?')">Limites</div>
    <div class="suggestion-chip" onclick="poserQuestion('Quelles sources de données ?')">Sources</div>
  </div>
  
  <div class="input-area">
    <input type="text" id="question" placeholder="Posez votre question..." 
           onkeypress="if(event.key==='Enter') envoyerQuestion()">
    <button onclick="envoyerQuestion()">Envoyer</button>
  </div>
  
  <div class="sources">
    <strong>Sources fiables :</strong>
    <a href="https://www.inrs.fr" target="_blank">INRS</a>
    <a href="https://dares.travail-emploi.gouv.fr" target="_blank">DARES</a>
    <a href="https://www.assurance-maladie.ameli.fr" target="_blank">Ameli</a>
    <a href="https://www.insee.fr" target="_blank">INSEE</a>
    <a href="https://www.has-sante.fr" target="_blank">HAS</a>
  </div>
</div>

<script>
const CONTEXTE = ''' + contexte_json + ''';

const REPONSES = {
  biais: "🔍 J'ai détecté " + CONTEXTE.biais_critiques + " biais critiques (écart > 10%) dans les données AT/MP par rapport à la population active INSEE. Le test du Chi² confirme un biais significatif dans 9 secteurs sur 9. Les 60+ ans sont sous-représentés de 30 à 50% dans les secteurs industriels (effet 'travailleur sain'). Les femmes sont sous-représentées dans les AT/MP du BTP et de la métallurgie.",
  modele: "🤖 J'utilise un Random Forest entraîné sur 151 codes NAF avec 15 features (10 expositions DARES + 5 démographiques). Accuracy " + CONTEXTE.accuracy_modele + ", écart fairness " + CONTEXTE.fairness + " (équité satisfaisante). L'explicabilité est assurée par SHAP : les contraintes posturales et les agents chimiques sont les features les plus influentes.",
  correlations: "📊 " + CONTEXTE.correlations_significatives + " corrélations significatives détectées (p < 0.05). Top 3 : contraintes posturales → indice fréquence AT (r=0.91), agents biologiques → MP (r=0.89), travail sur écran → AT (r=-0.86, effet protecteur). Voir Notebook 03 pour la heatmap.",
  limites: "⚠️ Trois limites principales : (1) dataset agrégé au niveau secteur — pas d'analyse par entreprise; (2) le biais déclaratif n'est pas corrigé — un secteur sous-déclarant apparaît à tort à faible risque; (3) périmètre régime général uniquement — pas de MSA, DOM, ou indépendants.",
  sources: "📚 Trois sources ouvertes : (1) Ameli — Open Data Risques Professionnels (AT/MP par CTN/NAF); (2) DARES — Enquête SUMER 2017 (expositions par secteur); (3) INSEE — Enquête Emploi (population active de référence).",
  defaut: "Je peux vous renseigner sur : les biais détectés, le modèle de Machine Learning, les corrélations AT × DARES, les sources de données, ou les limites du projet. Quelle dimension vous intéresse ?",
};

function poserQuestion(q) {
  document.getElementById('question').value = q;
  envoyerQuestion();
}

function envoyerQuestion() {
  const input = document.getElementById('question');
  const question = input.value.trim();
  if (!question) return;
  
  ajouterMessage(question, 'user');
  input.value = '';
  
  setTimeout(() => {
    const q = question.toLowerCase();
    let reponse;
    if (q.includes('biais') || q.includes('représentat')) reponse = REPONSES.biais;
    else if (q.includes('modèle') || q.includes('ml') || q.includes('machine learning') || q.includes('prédict')) reponse = REPONSES.modele;
    else if (q.includes('corrélat') || q.includes('correlation') || q.includes('relation')) reponse = REPONSES.correlations;
    else if (q.includes('limit') || q.includes('faiblesse') || q.includes('défaut')) reponse = REPONSES.limites;
    else if (q.includes('source') || q.includes('donnée') || q.includes('jeux')) reponse = REPONSES.sources;
    else reponse = REPONSES.defaut;
    ajouterMessage(reponse, 'bot');
  }, 400);
}

function ajouterMessage(texte, type) {
  const chat = document.getElementById('chat');
  const div = document.createElement('div');
  div.className = 'message message-' + type;
  
  if (type === 'bot') {
    div.innerHTML = '<div class="mini-avatar">LB</div><div class="bubble bubble-bot">' + texte + '</div>';
  } else {
    div.innerHTML = '<div class="bubble bubble-user">' + texte + '</div>';
  }
  
  chat.appendChild(div);
  chat.scrollTop = chat.scrollHeight;
}
</script>

</body>
</html>'''
    
    return html

# Générer et sauvegarder le widget HTML
html_widget = generer_widget_html()
os.makedirs('../assets', exist_ok=True)
with open('../assets/chatbot.html', 'w', encoding='utf-8') as f:
    f.write(html_widget)

print(f"Widget HTML généré : {len(html_widget)} caractères")
print(f"Sauvegardé dans : assets/chatbot.html")
print(f"\n→ Ce fichier peut être hébergé directement sur GitHub Pages")
print(f"→ Il sert de démonstration interactive pour les recruteurs")


# 7. Bonnes pratiques pour le déploiement

## Sécurité de la clé API

**NE JAMAIS** committer la clé API dans Git. Utilise :

```bash
# Dans un fichier .env (à ajouter à .gitignore)
ANTHROPIC_API_KEY=sk-ant-...

# Ou en variable d'environnement
export ANTHROPIC_API_KEY="sk-ant-..."
```

Dans le code :
```python
import os
from dotenv import load_dotenv
load_dotenv()
api_key = os.environ.get('ANTHROPIC_API_KEY')
```

## Architecture recommandée pour la production

- **Frontend statique** sur GitHub Pages (le `chatbot.html` généré ici)
- **Backend proxy** sur Make.com / Cloudflare Worker / Vercel
  pour cacher la clé API et limiter les abus (rate limiting)
- **Logging** des conversations pour amélioration continue (avec consentement RGPD)

## Évolutions possibles

1. RAG sur les notebooks complets (Vectorize les .ipynb → recherche sémantique)
2. Tool use : permettre au bot de relancer des analyses (recalculer un Chi²)
3. Intégration avec ton site portfolio existant (PréventIA-LaB)
4. Version vocale (Web Speech API + tts)


# 8. Résumé et clôture du projet


In [ ]:
print("=" * 70)
print("  PROJET PORTFOLIO — CLÔTURE")
print("=" * 70)
print(f"""
  📊 5 notebooks complets et exécutables :
     01_exploration.ipynb    — Profilage des données
     02_biais.ipynb          — Détection des biais (Chi², Shannon)
     03_correlations.ipynb   — Corrélations AT × DARES (Pearson)
     04_prediction_ml.ipynb  — Modèle ML (Random Forest, SHAP, fairness)
     05_chatbot.ipynb        — Chatbot d'exploration IA

  📈 Résultats clés du projet :
     • {len(df_biais)} biais critiques détectés (Notebook 02)
     • {len(df_correlations)} corrélations significatives (Notebook 03)
     • Modèle ML — Accuracy {resume_modele['accuracy_test']:.1%}, fairness {resume_modele['ecart_fairness']:.1%}
     • Chatbot avec contexte projet enrichi

  🎯 Compétences démontrées :
     ✓ Data Analysis (pandas, profiling)
     ✓ Data Visualization (matplotlib, seaborn, heatmaps)
     ✓ Statistiques (Chi², Pearson, Spearman, Shannon)
     ✓ Machine Learning (Random Forest, validation croisée)
     ✓ Explicabilité (SHAP, permutation importance)
     ✓ IA responsable (audit fairness, détection biais)
     ✓ Intégration API (Anthropic Claude)
     ✓ Expertise métier (20 ans en santé au travail)

  💼 Prêt pour ton portfolio GitHub et tes entretiens d'alternance !
""")
